## 12.6.3.1 Creating the endpoint

In [1]:
!pip install Flask

You should consider upgrading via the 'c:\Users\avrae\OneDrive\Desktop\Desktop\MLOps\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [13]:
import mlflow
import json
import pandas as pd
import numpy as np
from mlflow import MlflowClient
from flask import Flask, jsonify, request

In [14]:
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

In [15]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:8080")

client = mlflow.MlflowClient()

models = client.search_registered_models()

for model in models:
    print("Model:", model.name)

Model: logistic


In [16]:
model_versions = client.search_model_versions(
    "name='logistic'"
)

for mv in model_versions:
    print(
        "Version:", mv.version,
        "Status:", mv.status
    )

Version: 1 Status: READY


In [18]:
from flask import Flask

app = Flask(__name__)

print("Flask app created successfully")

Flask app created successfully


In [19]:
# Connect to MLflow server
mlflow.set_tracking_uri("http://127.0.0.1:8080")

# Registered model
model_name = "logistic"

# Model version
model_version = "1"

# Load model from MLflow Model Registry
app.model = mlflow.pyfunc.load_model(
    model_uri=f"models:/{model_name}/{model_version}"
)

print("Model loaded successfully!")

Model loaded successfully!


In [23]:
print(app.model)

mlflow.pyfunc.loaded_model:
  artifact_path: logreg
  flavor: mlflow.sklearn
  run_id: b6260ff6c22845f8844e3a734fb4f5fa



In [27]:
import requests

response = requests.get("http://127.0.0.1:8080/version")

print(response.text)

2.13.1


In [28]:
import mlflow

print("Client:", mlflow.__version__)

Client: 2.13.1


In [32]:
import json
import pandas as pd

data = """{
    "row.names": 1,
    "sbp": 160,
    "tobacco": 4.8,
    "ldl": 8.63,
    "adiposity": 36.21,
    "famhist": "Present",
    "typea": 50,
    "obesity": 34.72,
    "alcohol": 28.8,
    "age": 60
}"""

data_dict = json.loads(data)

data_df = pd.DataFrame([data_dict])

prediction = app.model.predict(data_df)

print(prediction)

[1]


In [34]:
@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()

    data_df = pd.DataFrame([data])

    prediction = app.model.predict(data_df)

    return jsonify({
        "prediction": prediction.tolist()
    })

In [35]:
app.run(port=5002,
        debug=True,
        use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5002
Press CTRL+C to quit
127.0.0.1 - - [07/Sep/2026 08:49:35] "POST /predict HTTP/1.1" 415 -
127.0.0.1 - - [07/Sep/2026 08:50:40] "POST /predict HTTP/1.1" 200 -
